# Exercício 3 — Padronização e Escalonamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

colunas = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status", "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]
arquivo = Path("adult.data")
if not arquivo.exists():
    arquivo = Path("adult.data")
df = pd.read_csv(arquivo, header=None, names=colunas, skipinitialspace=False, na_values=["?", " ?"])
df.head()

## Inspeção das colunas categóricas

In [ ]:
categoricas = df.select_dtypes(include="object").columns
antes = {coluna: int(df[coluna].nunique(dropna=False)) for coluna in categoricas}
valores_antes = {coluna: df[coluna].dropna().astype(str).unique()[:20].tolist() for coluna in categoricas}
valores_antes

## Padronização dos textos

In [ ]:
df_padronizado = df.copy()
for coluna in categoricas:
    df_padronizado[coluna] = df_padronizado[coluna].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.casefold()
depois = {coluna: int(df_padronizado[coluna].nunique(dropna=False)) for coluna in categoricas}
pd.DataFrame({"antes": pd.Series(antes), "depois": pd.Series(depois), "ausentes": df_padronizado.isna().sum()})

## Conversão dos marcadores de ausência

In [ ]:
ausentes_apareceram = df_padronizado.isna().sum()
ausentes_apareceram[ausentes_apareceram > 0]

## Escalonamento

In [ ]:
numericas = df_padronizado.select_dtypes(include=np.number).columns
base_numerica = df_padronizado[numericas].copy()
minmax = pd.DataFrame(MinMaxScaler().fit_transform(base_numerica), columns=numericas)
zscore = pd.DataFrame(StandardScaler().fit_transform(base_numerica), columns=numericas)
robusto = pd.DataFrame(RobustScaler().fit_transform(base_numerica), columns=numericas)
resumo_escalas = pd.concat({"minmax": minmax.agg(["min", "max", "mean", "std"]), "zscore": zscore.agg(["min", "max", "mean", "std"]), "robusto": robusto.agg(["min", "max", "mean", "std"])}, axis=1)
resumo_escalas

## Comparação visual

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(base_numerica["capital-gain"], bins=40, ax=eixos[0])
sns.histplot(zscore["capital-gain"], bins=40, ax=eixos[1])
sns.histplot(robusto["capital-gain"], bins=40, ax=eixos[2])
eixos[0].set_title("Original")
eixos[1].set_title("Z-score")
eixos[2].set_title("Robusto")
plt.tight_layout()
plt.show()

## Respostas de reflexão

O Z-score é mais interpretável quando a variável tem comportamento aproximadamente simétrico e pode ser comparada em relação a uma média e a um desvio-padrão. A variável price, em outros conjuntos, costuma ser assimétrica; por isso a distribuição deve ser observada antes de interpretar valores extremos apenas pelo Z-score.

Como capital-gain tem muitos zeros e poucos valores altos, Min-Max e Z-score são influenciados pelos extremos e comprimem a maior parte das observações perto de um único ponto. O escalonamento robusto usa mediana e intervalo interquartil, sendo menos sensível a esses valores.

Não é obrigatório escalonar toda coluna numérica. fnlwgt é um peso amostral que representa quantas pessoas da população são representadas por uma observação; tratá-lo como uma medida física comum pode não fazer sentido, dependendo do modelo e da finalidade.